# Hale (1908) — Magnetic Field in Sun-Spots: Implementation
# 흑점의 자기장: 구현

**Paper / 논문**: G.E. Hale, "On the Probable Existence of a Magnetic Field in Sun-Spots," *ApJ*, 28, 315–343, 1908.

이 노트북은 Hale의 1908년 논문의 핵심 물리학을 구현합니다:
This notebook implements the key physics of Hale's 1908 paper:

1. **Part 1**: Zeeman 효과 — 에너지 준위 분리와 스펙트럼 선 분리 시뮬레이션 / Zeeman Effect simulation
2. **Part 2**: Hale의 실험실 비교 — Tables I–III 데이터 재현 및 자기장 추정 / Laboratory comparison and field estimation
3. **Part 3**: 편광 시뮬레이션 — Fresnel rhomb + Nicol prism 효과 / Polarization simulation
4. **Part 4**: Stokes parameters와 Zeeman 프로파일 / Stokes profiles of Zeeman-split lines
5. **Part 5**: 자기장의 고도 의존성 모델 / Height dependence of magnetic field
6. **Part 6**: 현대 vector magnetogram 개념 시각화 / Modern vector magnetogram visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.constants import e, m_e, c, hbar, physical_constants

# Physical constants in CGS for consistency with Hale's units
e_cgs = 4.80326e-10      # electron charge (esu)
m_e_cgs = 9.10938e-28    # electron mass (g)
c_cgs = 2.99792e10       # speed of light (cm/s)
mu_B_cgs = e_cgs * hbar * 1e7 / (2 * m_e_cgs * c_cgs)  # Bohr magneton (erg/G)

# SI constants
mu_B_SI = physical_constants['Bohr magneton'][0]  # J/T

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

## Part 1: Zeeman Effect — Energy Level Splitting & Spectral Line Splitting
## 파트 1: Zeeman 효과 — 에너지 준위 분리와 스펙트럼 선 분리

자기장 $B$ 내에서 에너지 준위의 분리:
Energy level splitting in magnetic field $B$:

$$\Delta E = m_l \mu_B B$$

스펙트럼 선의 파장 분리 (Normal Zeeman Effect):
Wavelength splitting of spectral lines:

$$\Delta\lambda = \frac{e B \lambda^2}{4\pi m_e c^2}$$

먼저 에너지 준위 다이어그램을 시각화하고, 자기장 세기에 따른 스펙트럼 선 분리를 시뮬레이션합니다.
First we visualize the energy level diagram, then simulate spectral line splitting vs. field strength.

In [ ]:
def zeeman_splitting_angstrom(wavelength_A: float, B_gauss: float) -> float:
    """Calculate normal Zeeman splitting in Angstroms.

    Args:
        wavelength_A: Central wavelength in Angstroms.
        B_gauss: Magnetic field strength in Gauss.

    Returns:
        Wavelength splitting delta_lambda in Angstroms.
    """
    lam_cm = wavelength_A * 1e-8
    delta_lam = e_cgs * B_gauss * lam_cm**2 / (4 * np.pi * m_e_cgs * c_cgs**2)
    return delta_lam * 1e8  # Convert back to Angstroms


# --- Energy Level Diagram ---
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# Left: Energy level diagram
ax = axes[0]
ax.set_xlim(-0.5, 2.5)
ax.set_ylim(-0.5, 3.5)
ax.set_xticks([0.5, 1.8])
ax.set_xticklabels(['B = 0\n(degenerate)', 'B ≠ 0\n(split)'], fontsize=11)
ax.set_ylabel('Energy', fontsize=13)
ax.set_title('Zeeman Splitting: Energy Level Diagram\nZeeman 분리: 에너지 준위 다이어그램', fontsize=13)
ax.grid(False)

# Lower level: l=0, m_l=0
ax.hlines(0.5, -0.1, 1.1, colors='navy', linewidths=2.5)
ax.text(-0.3, 0.5, r'$l=0$', fontsize=12, va='center', color='navy')
ax.hlines(0.5, 1.3, 2.3, colors='navy', linewidths=2.5)
ax.text(2.35, 0.5, r'$m_l=0$', fontsize=11, va='center', color='navy')

# Upper level: l=1, m_l = -1, 0, +1
ax.hlines(2.5, -0.1, 1.1, colors='darkred', linewidths=2.5)
ax.text(-0.3, 2.5, r'$l=1$', fontsize=12, va='center', color='darkred')

spread = 0.6
for i, ml in enumerate([-1, 0, 1]):
    y = 2.5 + ml * spread
    ax.hlines(y, 1.3, 2.3, colors='darkred', linewidths=2.5)
    ax.text(2.35, y, f'$m_l={ml:+d}$', fontsize=11, va='center', color='darkred')

# Transition arrows
colors_trans = ['blue', 'green', 'red']
labels = [r'$\sigma^-$ (red shift)', r'$\pi$ (no shift)', r'$\sigma^+$ (blue shift)']
for i, (ml, col, lab) in enumerate(zip([-1, 0, 1], colors_trans, labels)):
    y_upper = 2.5 + ml * spread
    x_pos = 1.5 + i * 0.25
    ax.annotate('', xy=(x_pos, 0.6), xytext=(x_pos, y_upper - 0.1),
                arrowprops=dict(arrowstyle='->', color=col, lw=1.8))
    ax.text(x_pos, 1.5 + (i - 1) * 0.25, lab, fontsize=9, ha='center',
            color=col, rotation=0, bbox=dict(boxstyle='round,pad=0.15',
            fc='white', ec=col, alpha=0.8))

# Right: Spectral line splitting vs. B
ax2 = axes[1]
lam0 = 6302.71  # Hale's iron line
B_range = np.linspace(0, 5000, 500)
delta_lam = zeeman_splitting_angstrom(lam0, B_range)

ax2.plot(B_range, delta_lam, 'b-', lw=2, label=r'$\Delta\lambda$ (Normal Zeeman)')
ax2.axhline(0.252, color='red', ls='--', lw=1.5,
            label=f'Hale observed: 0.252 Å (→ ~3060 G)')
ax2.axvline(2900, color='orange', ls=':', lw=1.5,
            label='Hale estimate: ~2900 G')

ax2.set_xlabel('Magnetic Field B (Gauss)', fontsize=13)
ax2.set_ylabel(r'$\Delta\lambda$ (Å)', fontsize=13)
ax2.set_title(f'Zeeman Splitting vs. Field Strength\n'
              f'λ = {lam0} Å (Fe I, Hale\'s line)', fontsize=13)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

# Print key values
print("=== Zeeman Splitting Calculations ===")
for B in [1000, 2000, 2900, 3000, 5000, 15000]:
    dl = zeeman_splitting_angstrom(lam0, B)
    print(f"  B = {B:6d} G  →  Δλ = {dl:.4f} Å  (λ = {lam0} Å)")

## Part 2: Hale's Laboratory Comparison — Tables I–III
## 파트 2: Hale의 실험실 비교 — 표 I–III

Hale은 실험실 spark에서 측정한 Zeeman 분리(ΔΛ, Spark)와 흑점에서 관측한 분리(ΔΛ, Spot)를 비교했습니다.
비율의 일관성이 자기 기원의 증거입니다.

Hale compared laboratory Zeeman splitting (ΔΛ, Spark) with sunspot observations (ΔΛ, Spot).
Consistency of the ratio is evidence for magnetic origin.

$$B_{\text{spot}} = B_{\text{lab}} \times \frac{\Delta\lambda_{\text{spot}}}{\Delta\lambda_{\text{lab}}}$$

In [ ]:
# Hale's data from Tables I, II, III
# Table I: Iron doublets (B_lab = 15,000 G)
iron_data = {
    'wavelength': [6213.14, 6301.72, 6302.71, 6337.05],
    'dλ_spark':   [0.703,   0.737,   1.230,   0.895],
    'dλ_spot':    [0.136,   0.138,   0.252,   0.172],
    'B_lab': 15000,
    'element': 'Iron (Fe)'
}

# Table II: Titanium doublets (B_lab = 12,500 G)
titanium_data = {
    'wavelength': [5903.56, 5938.04, 6064.85, 6303.98, 6312.46],
    'dλ_spark':   [0.732,   0.737,   0.876,   0.493,   0.615],
    'dλ_spot':    [0.086,   0.080,   0.184,   0.093,   0.091],
    'B_lab': 12500,
    'element': 'Titanium (Ti)'
}

# Table III: Chromium doublets (B_lab = 12,500 G)
chromium_data = {
    'wavelength': [5304.36, 5387.16, 5713.00, 5781.40, 5781.97,
                   5783.29, 5784.08, 5785.19],
    'dλ_spark':   [0.636, 0.676, 0.610, 0.755, 0.922,
                   0.772, 0.720, 0.707],
    'dλ_spot':    [0.188, 0.085, 0.161, 0.121, 0.212,
                   0.137, 0.121, 0.137],
    'B_lab': 12500,
    'element': 'Chromium (Cr)'
}

# --- Compute ratios and estimated B_spot ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, data in zip(axes, [iron_data, titanium_data, chromium_data]):
    lam = np.array(data['wavelength'])
    spark = np.array(data['dλ_spark'])
    spot = np.array(data['dλ_spot'])
    ratio = spark / spot
    B_spot = data['B_lab'] / ratio

    ax.bar(range(len(lam)), ratio, color=['steelblue', 'coral', 'seagreen',
           'gold', 'plum', 'skyblue', 'salmon', 'lightgreen'][:len(lam)],
           edgecolor='black', alpha=0.8)
    ax.axhline(np.mean(ratio), color='red', ls='--', lw=2,
               label=f'Mean ratio = {np.mean(ratio):.1f}\n'
                     f'→ B_spot ≈ {data["B_lab"]/np.mean(ratio):.0f} G')
    ax.set_xticks(range(len(lam)))
    ax.set_xticklabels([f'{l:.0f}' for l in lam], rotation=45, fontsize=9)
    ax.set_xlabel('Wavelength (Å)', fontsize=11)
    ax.set_ylabel('ΔΛ_Spark / ΔΛ_Spot', fontsize=11)
    ax.set_title(f'{data["element"]}\n(B_lab = {data["B_lab"]:,} G)', fontsize=12)
    ax.legend(fontsize=10, loc='upper right')

plt.suptitle("Hale's Lab/Spot Splitting Ratios — Tables I, II, III\n"
             "Hale의 실험실/흑점 분리 비율", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Summary table
print("=" * 70)
print(f"{'Element':<12} {'B_lab (G)':>10} {'Mean Ratio':>12} {'B_spot (G)':>12}")
print("-" * 70)
for data in [iron_data, titanium_data, chromium_data]:
    ratio = np.mean(np.array(data['dλ_spark']) / np.array(data['dλ_spot']))
    B_est = data['B_lab'] / ratio
    print(f"{data['element']:<12} {data['B_lab']:>10,} {ratio:>12.1f} {B_est:>12,.0f}")
print("=" * 70)

## Part 3: Polarization Simulation — Fresnel Rhomb + Nicol Prism
## 파트 3: 편광 시뮬레이션 — Fresnel Rhomb + Nicol Prism

Hale의 관측을 시뮬레이션합니다: Zeeman doublet의 두 성분이 반대 방향 원편광을 가지며,
Nicol prism을 회전하면 한쪽 성분만 선택적으로 투과됩니다.

Simulating Hale's observation: the two Zeeman doublet components have opposite circular polarization,
and rotating the Nicol prism selectively transmits one component.

In [ ]:
def gaussian(x: np.ndarray, mu: float, sigma: float) -> np.ndarray:
    """Normalized Gaussian profile."""
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2)


def simulate_nicol_observation(lam0: float, delta_lam: float,
                               nicol_angle_deg: float,
                               sigma: float = 0.03) -> tuple:
    """Simulate what Hale saw through the Fresnel rhomb + Nicol.

    After the Fresnel rhomb, σ+ becomes horizontal polarization
    and σ- becomes vertical polarization. The Nicol transmits
    the component aligned with its axis.

    Args:
        lam0: Central wavelength (Å).
        delta_lam: Zeeman splitting (Å).
        nicol_angle_deg: Nicol prism angle (degrees).
        sigma: Line width (Å).

    Returns:
        Tuple of (wavelength array, transmitted spectrum).
    """
    lam = np.linspace(lam0 - 0.5, lam0 + 0.5, 1000)
    theta = np.radians(nicol_angle_deg)

    # σ+ component (blue-shifted) → horizontal after rhomb
    sigma_plus = gaussian(lam, lam0 - delta_lam / 2, sigma)
    # σ- component (red-shifted) → vertical after rhomb
    sigma_minus = gaussian(lam, lam0 + delta_lam / 2, sigma)

    # Nicol transmission: cos²θ for horizontal, sin²θ for vertical
    transmitted = sigma_plus * np.cos(theta)**2 + sigma_minus * np.sin(theta)**2

    return lam, transmitted


# --- Simulate Hale's Plate XXVI ---
lam0 = 5940.87  # Vanadium line from Plate XXVI
delta_lam = 0.15  # Approximate splitting

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Top row: Different Nicol angles for SAME spot (clockwise vortex)
angles_top = [0, 45, 90]
titles_top = ['Nicol 0° (σ⁺ only)\nRed component',
              'Nicol 45° (both equal)',
              'Nicol 90° (σ⁻ only)\nViolet component']

for ax, angle, title in zip(axes[0], angles_top, titles_top):
    lam, spec = simulate_nicol_observation(lam0, delta_lam, angle)

    # Background: absorption line
    solar = 1.0 - 0.5 * gaussian(lam, lam0, 0.04)
    # Spot spectrum: show as emission-like doublet for clarity
    ax.fill_between(lam, 0, spec, alpha=0.3, color='blue')
    ax.plot(lam, spec, 'b-', lw=2)

    # Mark components
    ax.axvline(lam0 - delta_lam/2, color='blue', ls=':', alpha=0.5, label='σ⁺ (violet)')
    ax.axvline(lam0 + delta_lam/2, color='red', ls=':', alpha=0.5, label='σ⁻ (red)')
    ax.axvline(lam0, color='gray', ls='--', alpha=0.3)

    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Wavelength (Å)')
    ax.set_ylabel('Intensity')
    ax.set_xlim(lam0 - 0.4, lam0 + 0.4)
    ax.legend(fontsize=8, loc='upper right')

# Bottom row: REVERSED polarity (counter-clockwise vortex)
# For opposite vortex, σ+ and σ- swap positions
for ax, angle, title in zip(axes[1], angles_top,
    ['Nicol 0° → Violet comp.\n(reversed!)',
     'Nicol 45° (both equal)',
     'Nicol 90° → Red comp.\n(reversed!)']):

    lam, spec = simulate_nicol_observation(lam0, delta_lam, 90 - angle)

    ax.fill_between(lam, 0, spec, alpha=0.3, color='red')
    ax.plot(lam, spec, 'r-', lw=2)
    ax.axvline(lam0 - delta_lam/2, color='blue', ls=':', alpha=0.5)
    ax.axvline(lam0 + delta_lam/2, color='red', ls=':', alpha=0.5)
    ax.axvline(lam0, color='gray', ls='--', alpha=0.3)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Wavelength (Å)')
    ax.set_ylabel('Intensity')
    ax.set_xlim(lam0 - 0.4, lam0 + 0.4)

axes[0][0].set_ylabel('Clockwise Vortex\n(시계방향 소용돌이)\nIntensity', fontsize=11)
axes[1][0].set_ylabel('Counter-clockwise\n(반시계방향)\nIntensity', fontsize=11)

plt.suptitle("Simulating Hale's Plate XXVI: Nicol Rotation & Vortex Polarity Reversal\n"
             "Hale의 Plate XXVI 시뮬레이션: Nicol 회전과 소용돌이 극성 반전",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Part 4: Stokes Parameters & Zeeman Profiles
## 파트 4: Stokes 매개변수와 Zeeman 프로파일

현대적 spectropolarimetry에서 Zeeman 분리된 선의 Stokes 프로파일을 시뮬레이션합니다.

Modern spectropolarimetry measures Zeeman-split lines as Stokes profiles:

- **Stokes $I$**: 총 강도 → 선의 넓어짐/분리 / Total intensity → line broadening/splitting
- **Stokes $V$**: 원편광 → $\propto g_{\text{eff}} B_\parallel \frac{dI}{d\lambda}$ (S자 형태) / Circular polarization (S-shaped)
- **Stokes $Q$, $U$**: 평면편광 → $\propto g_{\text{eff}}^2 B_\perp^2 \frac{d^2I}{d\lambda^2}$ / Linear polarization

In [ ]:
def stokes_profiles(lam: np.ndarray, lam0: float, B_gauss: float,
                    gamma_deg: float, chi_deg: float,
                    g_eff: float = 2.5, sigma: float = 0.05,
                    depth: float = 0.7) -> dict:
    """Compute approximate Stokes I, Q, U, V profiles for a Zeeman-split line.

    Uses the weak-field approximation for V and Q/U.

    Args:
        lam: Wavelength array (Å).
        lam0: Central wavelength (Å).
        B_gauss: Total magnetic field strength (Gauss).
        gamma_deg: Inclination angle (0=toward observer, 90=transverse).
        chi_deg: Azimuth angle of transverse field (degrees).
        g_eff: Effective Landé g-factor.
        sigma: Doppler width of the line (Å).
        depth: Line depth (0 to 1).

    Returns:
        Dictionary with 'I', 'Q', 'U', 'V' profiles.
    """
    gamma = np.radians(gamma_deg)
    chi = np.radians(chi_deg)

    # Zeeman splitting
    delta_lam_B = zeeman_splitting_angstrom(lam0, B_gauss) * g_eff

    # Unperturbed line profile (absorption)
    phi = gaussian(lam, lam0, sigma)

    # Stokes I: absorption line (slightly broadened by field)
    I_profile = 1.0 - depth * phi

    # Derivative of the profile (for Stokes V)
    dphi_dlam = -(lam - lam0) / sigma**2 * phi

    # Second derivative (for Stokes Q, U)
    d2phi_dlam2 = ((lam - lam0)**2 / sigma**4 - 1 / sigma**2) * phi

    # Stokes V ~ -g_eff * delta_lam_B * cos(gamma) * dI/dlam
    V_profile = depth * delta_lam_B * np.cos(gamma) * dphi_dlam

    # Stokes Q, U ~ g_eff^2 * delta_lam_B^2 * sin^2(gamma) * d2I/dlam2
    lin_amp = 0.25 * depth * delta_lam_B**2 * np.sin(gamma)**2
    Q_profile = lin_amp * np.cos(2 * chi) * d2phi_dlam2
    U_profile = lin_amp * np.sin(2 * chi) * d2phi_dlam2

    return {'I': I_profile, 'Q': Q_profile, 'U': U_profile, 'V': V_profile}


# --- Stokes profiles for different field configurations ---
lam0 = 6302.5  # Fe I line (used by both Hale and modern SDO/HMI)
lam = np.linspace(lam0 - 0.4, lam0 + 0.4, 1000)

fig, axes = plt.subplots(4, 3, figsize=(15, 12), sharex=True)

configs = [
    {'B': 2900, 'gamma': 0, 'chi': 0,
     'title': 'Longitudinal (γ=0°)\nB toward observer\n(Hale: disk center spot)'},
    {'B': 2900, 'gamma': 90, 'chi': 30,
     'title': 'Transverse (γ=90°, χ=30°)\nB in plane of sky\n(Hale: limb spot)'},
    {'B': 2900, 'gamma': 45, 'chi': 60,
     'title': 'Oblique (γ=45°, χ=60°)\nMixed longitudinal+transverse'},
]

stokes_names = ['I', 'Q', 'U', 'V']
stokes_colors = ['black', 'blue', 'green', 'red']
stokes_labels = [
    'Stokes I (total intensity / 총 강도)',
    'Stokes Q (linear 0°/90° / 평면편광)',
    'Stokes U (linear 45°/135° / 평면편광)',
    'Stokes V (circular / 원편광)',
]

for col, cfg in enumerate(configs):
    profiles = stokes_profiles(lam, lam0, cfg['B'], cfg['gamma'], cfg['chi'])
    for row, (name, color, label) in enumerate(
            zip(stokes_names, stokes_colors, stokes_labels)):
        ax = axes[row, col]
        ax.plot(lam, profiles[name], color=color, lw=2)
        ax.axhline(0 if name != 'I' else 1, color='gray', ls='--', alpha=0.3)
        ax.axvline(lam0, color='gray', ls=':', alpha=0.3)

        if col == 0:
            ax.set_ylabel(label, fontsize=10)
        if row == 0:
            ax.set_title(cfg['title'], fontsize=11)
        if row == 3:
            ax.set_xlabel('Wavelength (Å)')

plt.suptitle(f"Stokes Profiles of Zeeman-Split Fe I λ{lam0} Å  (B = 2900 G, g_eff = 2.5)\n"
             f"Zeeman 분리된 Fe I λ{lam0} Å의 Stokes 프로파일",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Part 5: Height Dependence of Magnetic Field
## 파트 5: 자기장의 고도 의존성 모델

Hale은 서로 다른 원소의 선이 서로 다른 높이에서 형성됨을 이용하여, 자기장이 높이에 따라 감소함을 추론했습니다.
현대적 모델에서는 자기장이 쌍극자(dipole)에 가까운 형태로 감소합니다:

Hale inferred field decrease with height using multi-element line formation.
Modern models show the field decreases approximately as a dipole:

$$B(h) = B_0 \left(\frac{R_{\text{spot}}}{R_{\text{spot}} + h}\right)^3$$

In [ ]:
# Height dependence model
# Hale's data: formation heights from flash spectrum analysis
elements = {
    'Vanadium (V)':  {'h_min': 160, 'h_max': 320,   'color': 'green'},
    'Iron (Fe)':     {'h_min': 320, 'h_max': 1610,  'color': 'steelblue'},
    'Chromium (Cr)': {'h_min': 160, 'h_max': 1930,  'color': 'orange'},
    'Titanium (Ti)': {'h_min': 160, 'h_max': 5640,  'color': 'red'},
    'Sodium (Na)':   {'h_min': 320, 'h_max': 8000,  'color': 'purple'},
}

# Dipole decay model
B0 = 2900  # Surface field (Gauss)
R_spot = 5000  # Effective sunspot radius (km)

h = np.linspace(0, 10000, 500)  # Height in km
B_dipole = B0 * (R_spot / (R_spot + h))**3
B_exp = B0 * np.exp(-h / 2000)  # Exponential model for comparison

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Left: Field decay models
ax1.plot(h, B_dipole, 'b-', lw=2.5, label='Dipole: $B_0(R/(R+h))^3$')
ax1.plot(h, B_exp, 'r--', lw=2, label='Exponential: $B_0 e^{-h/2000}$')

# Mark element formation height ranges
for name, data in elements.items():
    h_mid = (data['h_min'] + data['h_max']) / 2
    B_mid = B0 * (R_spot / (R_spot + h_mid))**3
    ax1.barh(B_mid, data['h_max'] - data['h_min'],
             left=data['h_min'], height=80, color=data['color'],
             alpha=0.4, edgecolor=data['color'])
    ax1.text(data['h_max'] + 100, B_mid, name, fontsize=9,
             va='center', color=data['color'], fontweight='bold')

ax1.set_xlabel('Height above photosphere (km)', fontsize=12)
ax1.set_ylabel('Magnetic Field B (Gauss)', fontsize=12)
ax1.set_title('Magnetic Field Decay with Height\n자기장의 고도에 따른 감소', fontsize=13)
ax1.legend(fontsize=11)
ax1.set_xlim(0, 10000)

# Right: Implied Zeeman splitting vs height for Fe I 6302
ax2.plot(h, zeeman_splitting_angstrom(6302.71, B_dipole), 'b-', lw=2.5,
         label='Dipole model')
ax2.axhline(0.252, color='red', ls='--', lw=1.5,
            label='Hale observed (Fe): 0.252 Å')
ax2.axhline(0.05, color='gray', ls=':', lw=1,
            label='Detection limit (~0.05 Å)')

# Mark where Na D lines form (weak splitting)
h_na = 5000
B_na = B0 * (R_spot / (R_spot + h_na))**3
dl_na = zeeman_splitting_angstrom(5896, B_na)
ax2.plot(h_na, dl_na, 's', color='purple', ms=10, zorder=5)
ax2.annotate(f'Na D at h≈{h_na} km\nΔλ≈{dl_na:.3f} Å\n(barely detectable)',
             xy=(h_na, dl_na), xytext=(h_na + 800, dl_na + 0.04),
             fontsize=9, color='purple',
             arrowprops=dict(arrowstyle='->', color='purple'))

ax2.set_xlabel('Height above photosphere (km)', fontsize=12)
ax2.set_ylabel('Δλ for Fe I 6302.71 Å (Å)', fontsize=12)
ax2.set_title('Zeeman Splitting vs. Height\nZeeman 분리의 고도 의존성', fontsize=13)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("Hale's key observation: Na D and Mg b lines (forming at h > 5000 km)")
print("show very small Zeeman effect → field drops rapidly with height.")

## Part 6: Synthetic Vector Magnetogram
## 파트 6: 합성 벡터 자기도 시각화

현대 SDO/HMI가 생성하는 vector magnetogram의 개념을 시각화합니다.
Stokes $V$에서 시선 방향 성분(line-of-sight magnetogram)을, Stokes $Q$, $U$에서 횡방향 자기장을 결정합니다.

Visualizing the concept of a vector magnetogram as produced by modern SDO/HMI.
Stokes $V$ determines the line-of-sight component, Stokes $Q$, $U$ the transverse field.

In [ ]:
# Create a synthetic bipolar sunspot pair (simplified vector magnetogram)
nx, ny = 200, 200
x = np.linspace(-10, 10, nx)
y = np.linspace(-10, 10, ny)
X, Y = np.meshgrid(x, y)

# Two magnetic poles (bipolar sunspot pair)
# Positive polarity (leading spot, B toward observer)
x1, y1, B1 = -3, 0, 2500
# Negative polarity (following spot, B away from observer)
x2, y2, B2 = 3, 0, -2000

sigma_spot = 2.0

# Line-of-sight magnetic field (what Stokes V measures)
Bz = (B1 * np.exp(-((X - x1)**2 + (Y - y1)**2) / (2 * sigma_spot**2)) +
      B2 * np.exp(-((X - x2)**2 + (Y - y2)**2) / (2 * sigma_spot**2)))

# Transverse field (simplified potential field approximation)
# Gradient of Bz gives approximate horizontal field direction
Bx = -np.gradient(Bz, x, axis=1) * 5
By = -np.gradient(Bz, y, axis=0) * 5
B_transverse = np.sqrt(Bx**2 + By**2)

# --- Plot ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# 1. LOS Magnetogram (from Stokes V)
im1 = axes[0].imshow(Bz, extent=[-10, 10, -10, 10], cmap='RdBu_r',
                      vmin=-2500, vmax=2500, origin='lower')
axes[0].set_title('LOS Magnetogram (Stokes V)\n시선 방향 자기도 (Stokes V)', fontsize=12)
axes[0].set_xlabel('x (Mm)')
axes[0].set_ylabel('y (Mm)')
plt.colorbar(im1, ax=axes[0], label='$B_{\\parallel}$ (Gauss)', shrink=0.8)
axes[0].contour(X, Y, Bz, levels=[0], colors='black', linewidths=1)
axes[0].text(-3, -4, '+', fontsize=20, ha='center', color='white', fontweight='bold')
axes[0].text(3, -4, '−', fontsize=20, ha='center', color='white', fontweight='bold')

# 2. Transverse Field (from Stokes Q, U)
im2 = axes[1].imshow(B_transverse, extent=[-10, 10, -10, 10], cmap='hot',
                      vmin=0, vmax=1500, origin='lower')
# Quiver plot for field direction
skip = 12
axes[1].quiver(X[::skip, ::skip], Y[::skip, ::skip],
               Bx[::skip, ::skip], By[::skip, ::skip],
               color='cyan', alpha=0.8, scale=8000)
axes[1].set_title('Transverse Field (Stokes Q, U)\n횡방향 자기장 (Stokes Q, U)', fontsize=12)
axes[1].set_xlabel('x (Mm)')
plt.colorbar(im2, ax=axes[1], label='$B_{\\perp}$ (Gauss)', shrink=0.8)

# 3. Total field strength (from Stokes I broadening)
B_total = np.sqrt(Bz**2 + B_transverse**2)
im3 = axes[2].imshow(B_total, extent=[-10, 10, -10, 10], cmap='inferno',
                      vmin=0, vmax=3000, origin='lower')
axes[2].set_title('Total Field |B| (Stokes I)\n총 자기장 (Stokes I)', fontsize=12)
axes[2].set_xlabel('x (Mm)')
plt.colorbar(im3, ax=axes[2], label='$|B|$ (Gauss)', shrink=0.8)

plt.suptitle("Synthetic Vector Magnetogram of a Bipolar Sunspot Pair\n"
             "쌍극 흑점 쌍의 합성 벡터 자기도", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Modern instruments (SDO/HMI, Hinode/SP) produce images like these")
print("for the ENTIRE solar disk, using the Fe I 6173 Å or 6302 Å lines —")
print("the very same spectral lines Hale first analyzed in 1908!")

## Summary / 요약

| Part | 내용 / Content | Hale (1908) 연결 / Connection |
|---|---|---|
| 1 | Zeeman 효과 에너지 준위 분리 시뮬레이션 | 논문의 물리적 기초 — Zeeman (1896)의 발견 적용 |
| 2 | Tables I–III 데이터 재현, 자기장 추정 | 논문의 정량적 핵심 — B ≈ 2,600–2,900 G |
| 3 | Fresnel rhomb + Nicol 편광 관측 시뮬레이션 | Plate XXVI 재현 — 원편광 감지와 극성 반전 |
| 4 | Stokes parameters와 Zeeman 프로파일 | 현대적 확장 — Hale → SDO/HMI spectropolarimetry |
| 5 | 자기장 고도 의존성 모델 | 다중 원소 선 형성 높이를 이용한 수직 구조 추론 |
| 6 | 합성 vector magnetogram 시각화 | Hale의 Fe I 6302 Å → 현대 SDO/HMI의 동일 선 사용 |

**다음 논문 / Next paper**: Evershed (1909) — 흑점 반암부의 방사상 물질 흐름 발견.
Hale이 밝힌 자기장 위에 Evershed가 물질 흐름을 추가하여, 흑점의 MHD 구조를 완성합니다.

**Next paper**: Evershed (1909) — discovery of radial material flow in sunspot penumbrae.
Evershed adds mass flow atop Hale's magnetic field, completing the MHD picture of sunspots.